In [1]:
!pip install -q --disable-pip-version-check langchain langchain-groq gspread pydantic python-dotenv

In [2]:
import os
import csv
import time
from getpass import getpass
from dotenv import load_dotenv, find_dotenv

# 1. Load local .env if present
load_dotenv(find_dotenv())

# 2. Fallback to Google Colab Secrets if running in Colab
IN_COLAB = False
try:
    from google.colab import userdata, auth
    IN_COLAB = True
    if not os.environ.get("GROQ_API_KEY"):
        key = userdata.get("GROQ_API_KEY")
        if key:
            os.environ["GROQ_API_KEY"] = key
except Exception:
    pass

# 3. Interactive fallback if still missing
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter your GROQ_API_KEY: ").strip()

from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.2, max_retries=3, timeout=30)

def invoke_retry(runnable, payload, retries: int = 4):
    """Invoke a LangChain agent or structured runnable with exponential backoff on transient errors."""
    for attempt in range(retries):
        try:
            return runnable.invoke(payload)
        except (KeyError, ValueError, TypeError):
            raise
        except Exception as err:
            if attempt == retries - 1:
                raise
            wait_s = 2 ** (attempt + 1)
            print(f"   ⏳ Transient error ({type(err).__name__}), retrying in {wait_s}s...")
            time.sleep(wait_s)

print("✅ ChatGroq ready: openai/gpt-oss-20b (temperature=0.2)")

✅ ChatGroq ready: openai/gpt-oss-20b (temperature=0.2)


In [3]:
class LocalWorksheet:
    """Local CSV-backed worksheet mirror with the same append_row / get_all_records interface as gspread."""
    def __init__(self, csv_path: str, headers: list[str]):
        self.csv_path = csv_path
        self.headers = headers
        with open(self.csv_path, "w", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow(self.headers)

    def append_row(self, row: list):
        with open(self.csv_path, "a", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow(row)

    def get_all_records(self) -> list[dict]:
        with open(self.csv_path, "r", newline="", encoding="utf-8") as f:
            return list(csv.DictReader(f))


_gc = None

def init_sheet(sheet_title: str, headers: list[str], local_csv: str):
    """Open or create a Google Sheet in Colab; fall back to LocalWorksheet when running locally."""
    global _gc
    if IN_COLAB:
        import gspread
        from google.auth import default
        if _gc is None:
            auth.authenticate_user()
            creds, _ = default()
            _gc = gspread.authorize(creds)
        try:
            sh = _gc.open(sheet_title)
        except gspread.SpreadsheetNotFound:
            sh = _gc.create(sheet_title)
        ws_remote = sh.sheet1
        if not ws_remote.row_values(1):
            ws_remote.append_row(headers)
        print(f"✅ Google Sheet ready ({sheet_title}): {sh.url}")
        return ws_remote

    ws_local = LocalWorksheet(local_csv, headers)
    print(f"✅ Local Sheet mirror ready ({sheet_title}): {os.path.abspath(local_csv)}")
    return ws_local


ws = init_sheet(
    sheet_title="BrightPage Stationery Orders",
    headers=["customer_name", "item", "qty", "delivery_type", "delivery_date",
             "unit_price", "discount", "delivery_fee", "total", "notes"],
    local_csv="brightpage_orders.csv",
)

✅ Local Sheet mirror ready (BrightPage Stationery Orders): C:\Users\ramuv\AgenticAI\DAY4\brightpage_orders.csv


In [4]:
from typing import Literal, Optional
from pydantic import BaseModel, Field
from langchain.tools import tool

STATIONERY_CATALOGUE = {
    "spiral notebook (200 pgs)": 85,
    "a4 copier paper ream (500 sheets)": 320,
    "gel pen pack (10 pens, blue/black)": 150,
    "pastel highlighter set (6 colors)": 180,
    "heavy-duty stapler + pins": 240,
    "sticky notes pack (400 sheets)": 95,
    "geometry box set": 160,
    "whiteboard marker set (4 colors)": 120,
    "permanent marker (black)": 35,
    "clear file folder (pack of 5)": 110,
}

BRIGHTPAGE_HOURS = {
    "monday": "9:00 AM to 8:30 PM", "tuesday": "9:00 AM to 8:30 PM",
    "wednesday": "9:00 AM to 8:30 PM", "thursday": "9:00 AM to 8:30 PM",
    "friday": "9:00 AM to 8:30 PM", "saturday": "9:00 AM to 8:30 PM",
    "sunday": "10:00 AM to 4:00 PM",
}

class StationeryOrder(BaseModel):
    """A structured order taken from a BrightPage Stationery customer's message."""
    customer_name: str = Field(description="Customer's name")
    item: Literal[
        "spiral notebook (200 pgs)",
        "a4 copier paper ream (500 sheets)",
        "gel pen pack (10 pens, blue/black)",
        "pastel highlighter set (6 colors)",
        "heavy-duty stapler + pins",
        "sticky notes pack (400 sheets)",
        "geometry box set",
        "whiteboard marker set (4 colors)",
        "permanent marker (black)",
        "clear file folder (pack of 5)",
    ] = Field(description="Closest matching item from the BrightPage Stationery catalogue")
    qty: int = Field(description="Quantity ordered", ge=1)
    delivery_type: Literal["home_delivery", "store_pickup"] = Field(
        description="Whether the customer wants home_delivery or store_pickup"
    )
    delivery_date: str = Field(description="Requested delivery or pickup date as the customer said it")
    notes: Optional[str] = Field(default=None, description="Optional delivery or packing notes")

In [5]:
def _resolve_stationery_item(query: str) -> tuple[str | None, int | None]:
    """Find exact or partial catalogue match for a stationery item name."""
    cleaned = query.lower().strip()
    if cleaned in STATIONERY_CATALOGUE:
        return cleaned, STATIONERY_CATALOGUE[cleaned]
    for cat_item, price in STATIONERY_CATALOGUE.items():
        if cleaned in cat_item or cat_item in cleaned:
            return cat_item, price
    tokens = [t.rstrip("s") for t in cleaned.split() if len(t) > 1]
    best_item, best_score = None, 0
    for cat_item in STATIONERY_CATALOGUE:
        score = sum(1 for tok in tokens if tok in cat_item)
        if score > best_score:
            best_item, best_score = cat_item, score
    if best_score >= 2:
        return best_item, STATIONERY_CATALOGUE[best_item]
    return None, None

@tool
def get_stationery_price(item: str, qty: int = 1) -> str:
    """Look up a stationery item in the BrightPage catalogue and calculate total price in rupees.
    Applies a 10% bulk discount automatically when quantity is 10 or more, or subtotal exceeds Rs 1000.

    Args:
        item: Stationery item name, e.g. 'spiral notebook (200 pgs)', 'a4 copier paper ream (500 sheets)', 'geometry box set'
        qty: Quantity requested (must be at least 1)
    """
    if qty < 1:
        return f"Minimum quantity is at least 1 (got {qty}). Ask the customer for a valid quantity."
    matched_item, unit_price = _resolve_stationery_item(item)
    if matched_item is None:
        return f"'{item}' is not in our catalogue. I'll check with the store manager about availability."
    subtotal = unit_price * qty
    discount = round(subtotal * 0.10, 2) if (qty >= 10 or subtotal > 1000) else 0.0
    final_total = subtotal - discount
    if discount > 0:
        return (
            f"{qty} x {matched_item} @ Rs {unit_price} = Rs {subtotal}, "
            f"10% bulk discount (-Rs {discount:.0f}) -> Rs {final_total:.0f}"
        )
    return f"{qty} x {matched_item} @ Rs {unit_price} = Rs {final_total:.0f}"

@tool
def is_store_open(day: str) -> str:
    """Check BrightPage Stationery's opening hours for a day of the week, e.g. 'sunday' or 'monday'."""
    hours = BRIGHTPAGE_HOURS.get(day.lower().strip())
    return f"Open {hours} on {day}." if hours else f"'{day}' is not a valid day of the week."

@tool
def check_delivery_fee(order_total: float, delivery_type: str = "home_delivery") -> str:
    """Check BrightPage Stationery's local delivery fee for an order total.
    Store pickup is always free; local home delivery is free for orders above Rs 500 and Rs 40 otherwise.

    Args:
        order_total: Order amount in rupees after any bulk discount
        delivery_type: Either 'home_delivery' or 'store_pickup'
    """
    if "pick" in delivery_type.lower():
        return "Store pickup is free (Rs 0)."
    if order_total > 500:
        return f"Delivery is free (Rs 0) because the order total (Rs {order_total:.0f}) is above Rs 500."
    return f"Local delivery fee is Rs 40 because the order total (Rs {order_total:.0f}) is Rs 500 or below."

@tool
def log_stationery_order(
    customer_name: str,
    item: str,
    qty: int,
    delivery_type: str,
    delivery_date: str,
    notes: str = "",
) -> str:
    """Save a confirmed BrightPage Stationery order to the Orders sheet.
    Use ONLY when customer_name, catalogue item, qty (>= 1), delivery_type ('home_delivery' or 'store_pickup'), and delivery_date are confirmed."""
    matched_item, unit_price = _resolve_stationery_item(item)
    if matched_item is None:
        return f"Could not save order: '{item}' is not in the BrightPage catalogue."
    dtype = delivery_type.lower().strip().replace(" ", "_")
    if dtype not in ("home_delivery", "store_pickup"):
        dtype = "home_delivery" if "deliver" in dtype else "store_pickup" if "pick" in dtype else dtype
    try:
        record = StationeryOrder(
            customer_name=customer_name.strip(),
            item=matched_item,
            qty=qty,
            delivery_type=dtype,
            delivery_date=delivery_date.strip(),
            notes=notes.strip() or None,
        )
    except Exception as e:
        return f"Could not save order: {e}. Ask the customer to correct it."

    subtotal = unit_price * record.qty
    discount = round(subtotal * 0.10, 2) if (record.qty >= 10 or subtotal > 1000) else 0.0
    after_discount = subtotal - discount
    delivery_fee = 0 if (record.delivery_type == "store_pickup" or after_discount > 500) else 40
    total = after_discount + delivery_fee

    ws.append_row([
        record.customer_name, record.item, record.qty, record.delivery_type,
        record.delivery_date, unit_price, discount, delivery_fee, total,
        record.notes or "",
    ])
    return (
        f"Saved order for {record.customer_name}: {record.qty} x {record.item} "
        f"({record.delivery_type} on {record.delivery_date}). "
        f"Discount: Rs {discount:.0f}, Delivery: Rs {delivery_fee}, Total: Rs {total:.0f}."
    )

In [6]:
from langchain.agents import create_agent

BRIGHTPAGE_SYSTEM = """ROLE:   You are the friendly order assistant for BrightPage Stationery & Office Supplies.
FACTS:  Catalogue items: spiral notebook (200 pgs), a4 copier paper ream (500 sheets), gel pen pack (10 pens, blue/black),
        pastel highlighter set (6 colors), heavy-duty stapler + pins, sticky notes pack (400 sheets), geometry box set,
        whiteboard marker set (4 colors), permanent marker (black), clear file folder (pack of 5).
        Bulk discount: 10% off on 10+ identical items or orders above Rs 1000.
        Delivery: Free local delivery above Rs 500; Rs 40 fee at or below Rs 500. Delivery is only within local city limits.
        Returns: Damaged items replaced within 3 days with receipt.
RULES:  ALWAYS use get_stationery_price for item checks and prices, is_store_open for timings, and check_delivery_fee for delivery charges; NEVER guess them.
        If an item is not in our catalogue, NEVER invent a price or accept the order — say "I'll check with the store manager about availability."
        Match clear customer wording (e.g. "spiral notebooks" -> "spiral notebook (200 pgs)") directly without asking for exact parenthetical SKU strings.
        If quantity is less than 1, always say "minimum quantity is at least 1" and ask how many they want.
        Before log_stationery_order, confirm customer_name, item, qty (>= 1), delivery_type, and delivery_date; ask if any is missing.
        After saving, read back the order summary and total.
FORMAT: Under 65 words, friendly, English or Telugu matching the customer, plain numbers without commas (e.g. Rs 1000)."""

brightpage_agent = create_agent(
    model=llm,
    tools=[get_stationery_price, is_store_open, check_delivery_fee, log_stationery_order],
    system_prompt=BRIGHTPAGE_SYSTEM,
)

history = []
def chat(text: str) -> str:
    global history
    try:
        result = invoke_retry(brightpage_agent, {"messages": history + [{"role": "user", "content": text}]})
        history = result["messages"]
        return history[-1].content
    except Exception as e:
        print("Agent error:", e)
        return "Sorry, I'm a little busy. Please try again in a minute."

print("Turn 1:", chat("Are you open on Sunday? How much are 2 A4 copier paper reams with home delivery?"))
print("Turn 2:", chat("I'm Ravi. Please book 2 A4 copier paper reams for home delivery on 24 Dec."))
print("\nBrightPage Sheet rows:", ws.get_all_records())

Turn 1: Yes, we’re open Sunday 10:00 AM‑4:00 PM.  
2 a4 copier paper reams cost Rs 640.  
Since the order is above Rs 500, home delivery is free.  

Could you please give me your name and the delivery date?


Turn 2: Order confirmed!  
Ravi – 2 a4 copier paper reams (home delivery on 24 Dec).  
Subtotal Rs 640, no discount, free delivery. Total Rs 640.  
Thanks for shopping with BrightPage!

BrightPage Sheet rows: [{'customer_name': 'Ravi', 'item': 'a4 copier paper ream (500 sheets)', 'qty': '2', 'delivery_type': 'home_delivery', 'delivery_date': '24 Dec', 'unit_price': '320', 'discount': '0.0', 'delivery_fee': '0', 'total': '640.0', 'notes': ''}]


In [7]:
from langchain_core.messages import AIMessage, ToolMessage

trace_result = invoke_retry(brightpage_agent, {
    "messages": [{"role": "user", "content": "Are you open on Sunday? How much are 2 A4 copier paper reams?"}]
})

print("=== Full Message Trace (pretty_print) ===")
for m in trace_result["messages"]:
    m.pretty_print()

model_calls = sum(1 for m in trace_result["messages"] if isinstance(m, AIMessage))
tool_calls = sum(len(m.tool_calls) for m in trace_result["messages"] if isinstance(m, AIMessage) and m.tool_calls)
tool_responses = sum(1 for m in trace_result["messages"] if isinstance(m, ToolMessage))
invoked_names = [
    tc["name"]
    for m in trace_result["messages"] if isinstance(m, AIMessage) and m.tool_calls
    for tc in m.tool_calls
]

print("\n=== Trace Summary ===")
print(f"Model calls (AIMessage):     {model_calls}")
print(f"Tool calls requested:        {tool_calls} -> {invoked_names}")
print(f"Tool results (ToolMessage):  {tool_responses}")

=== Full Message Trace (pretty_print) ===
================================ Human Message =================================

Are you open on Sunday? How much are 2 A4 copier paper reams?
================================== Ai Message ==================================
Tool Calls:
  is_store_open (fc_7ca8b650-203b-40f6-b668-682c22f1a152)
 Call ID: fc_7ca8b650-203b-40f6-b668-682c22f1a152
  Args:
    day: sunday
================================= Tool Message =================================
Name: is_store_open

Open 10:00 AM to 4:00 PM on sunday.
================================== Ai Message ==================================
Tool Calls:
  get_stationery_price (fc_5e0c916d-427a-4f0e-8c7b-c099c769a187)
 Call ID: fc_5e0c916d-427a-4f0e-8c7b-c099c769a187
  Args:
    item: a4 copier paper ream (500 sheets)
    qty: 2
================================= Tool Message =================================
Name: get_stationery_price

2 x a4 copier paper ream (500 sheets) @ Rs 320 = Rs 640
===============

In [8]:
structured_extractor = llm.with_structured_output(StationeryOrder)

messy_messages = [
    "yo im kiran, send 3 a4 paper reams to my flat tomorrow pls urgent",
    "hi!! anu here.. need 12 spiral notebooks 200pgs, will pick up from shop friday evening",
    "pls book 2 packs of those pastel highlighters for Suresh, deliver on 28 Nov, call before coming",
    "hey need 5 gel pen packs blue/black, name is Meera, store pickup saturday morning",
    "send 1 heavy duty stapler with pins to Ravi office on Monday, pack nicely thanks",
]

print("=== Testing StationeryOrder with_structured_output on 5 Messy Messages ===\n")
for idx, msg in enumerate(messy_messages, start=1):
    parsed = invoke_retry(structured_extractor, msg)
    print(f"Message {idx}: {msg}")
    print(f"Parsed  {idx}: {parsed.model_dump()}\n")

=== Testing StationeryOrder with_structured_output on 5 Messy Messages ===



Message 1: yo im kiran, send 3 a4 paper reams to my flat tomorrow pls urgent
Parsed  1: {'customer_name': 'Kiran', 'item': 'a4 copier paper ream (500 sheets)', 'qty': 3, 'delivery_type': 'home_delivery', 'delivery_date': 'tomorrow', 'notes': None}



Message 2: hi!! anu here.. need 12 spiral notebooks 200pgs, will pick up from shop friday evening
Parsed  2: {'customer_name': 'Anu', 'item': 'spiral notebook (200 pgs)', 'qty': 12, 'delivery_type': 'store_pickup', 'delivery_date': 'Friday evening', 'notes': None}



Message 3: pls book 2 packs of those pastel highlighters for Suresh, deliver on 28 Nov, call before coming
Parsed  3: {'customer_name': 'Suresh', 'item': 'pastel highlighter set (6 colors)', 'qty': 2, 'delivery_type': 'home_delivery', 'delivery_date': '28 Nov', 'notes': 'call before coming'}



Message 4: hey need 5 gel pen packs blue/black, name is Meera, store pickup saturday morning
Parsed  4: {'customer_name': 'Meera', 'item': 'gel pen pack (10 pens, blue/black)', 'qty': 5, 'delivery_type': 'store_pickup', 'delivery_date': 'Saturday morning', 'notes': None}



Message 5: send 1 heavy duty stapler with pins to Ravi office on Monday, pack nicely thanks
Parsed  5: {'customer_name': 'Ravi', 'item': 'heavy-duty stapler + pins', 'qty': 1, 'delivery_type': 'home_delivery', 'delivery_date': 'Monday', 'notes': 'pack nicely'}



In [9]:
rows_before = ws.get_all_records()

edge_cases = [
    ("Not in catalogue (scientific calculator)", "Can I order 2 scientific calculators for tomorrow?"),
    ("Missing delivery date", "I'm Sita. Please book 2 spiral notebooks for home delivery."),
    ("Zero quantity (0 geometry box sets)", "I'm Kiran. Please order 0 geometry box sets for 25 Dec."),
]

for label, prompt_text in edge_cases:
    res = invoke_retry(brightpage_agent, {"messages": [{"role": "user", "content": prompt_text}]})
    reply = res["messages"][-1].content
    print(f"[{label}]")
    print(f"  Customer: {prompt_text}")
    print(f"  Agent:    {reply}\n")

rows_after = ws.get_all_records()
print("Sheet rows before edge tests:", len(rows_before))
print("Sheet rows after edge tests: ", len(rows_after))
assert rows_before == rows_after, "Unexpected row written to sheet during invalid/incomplete tests!"
print("✅ Verified: Nothing wrong was written to the BrightPage sheet.")

[Not in catalogue (scientific calculator)]
  Customer: Can I order 2 scientific calculators for tomorrow?
  Agent:    I’ll check with the store manager about availability.



[Missing delivery date]
  Customer: I'm Sita. Please book 2 spiral notebooks for home delivery.
  Agent:    Sure, Sita! I’ve calculated the cost: 2 spiral notebooks (200 pgs) total Rs 170. For home delivery (order ≤ Rs 500) the fee is Rs 40, making the final amount Rs 210. Could you let me know the delivery date you’d like? Once I have that, I’ll log the order.



[Zero quantity (0 geometry box sets)]
  Customer: I'm Kiran. Please order 0 geometry box sets for 25 Dec.
  Agent:    Hi Kiran! Minimum quantity is at least 1. How many geometry box sets would you like to order? Also, would you prefer home delivery or store pickup?

Sheet rows before edge tests: 1
Sheet rows after edge tests:  1
✅ Verified: Nothing wrong was written to the BrightPage sheet.


In [10]:
tests_csv_path = os.path.join("..", "DAY3", "brightpage_tests.csv")
with open(tests_csv_path, encoding="utf-8") as f:
    test_rows = list(csv.DictReader(f))

print(f"Running {len(test_rows)} tests from {tests_csv_path} against Day 4 BrightPage agent...\n")
passed = 0
for row in test_rows:
    res = invoke_retry(brightpage_agent, {"messages": [{"role": "user", "content": row["question"]}]})
    ans = res["messages"][-1].content
    ans_norm = " ".join(ans.replace("\u202f%", "%").replace("\xa0%", "%").replace("\u202f", " ").replace("\xa0", " ").split()).lower()
    expected = row["must_include"].lower()
    ok = expected in ans_norm
    passed += int(ok)
    status = "PASS" if ok else "FAIL"
    print(f"{status} | {row['question']} -> {ans[:75]}")
    time.sleep(0.3)

print(f"\n{passed}/{len(test_rows)} passed ({passed / len(test_rows):.0%})")

Running 13 tests from ..\DAY3\brightpage_tests.csv against Day 4 BrightPage agent...



PASS | How much is a spiral notebook? -> A spiral notebook (200 pgs) costs Rs 85.


PASS | What is the price of 2 A4 copier paper reams? -> The price for 2 a4 copier paper reams (500 sheets each) is Rs 640.


PASS | Are you open on Sunday? -> Yes! We’re open on Sunday from 10:00 AM to 4:00 PM. Let me know what you’d 


PASS | Is delivery free for an order of Rs 600? -> Yes! For a Rs 600 order, local home delivery is free (Rs 0).


PASS | What is the delivery fee for a Rs 300 order? -> The local home delivery fee for a Rs 300 order is Rs 40.


... (5 outputs left out)

PASS | Can I order 2 scientific calculators for tomorrow? -> I’ll check with the store manager about availability.


PASS | I'm Sita. Please book 2 spiral notebooks for home delivery. -> Sure, Sita! I’ve got 2 spiral notebooks (200 pgs) ready for home delivery. 


PASS | I'm Kiran. Please order 0 geometry box sets for 25 Dec. -> Hi Kiran! Minimum quantity is at least 1. How many geometry box sets would 



13/13 passed (100%)


In [11]:
@tool
def tool1(item: str, qty: int = 1) -> str:
    """does stuff"""
    matched_item, unit_price = _resolve_stationery_item(item)
    if matched_item is None:
        return f"'{item}' is not in our catalogue."
    return f"{qty} x {matched_item} = Rs {unit_price * qty}"

SYSTEM_GENERIC = """ROLE:   You are the friendly order assistant for BrightPage Stationery & Office Supplies.
FACTS:  Orders are delivered locally within city limits.
RULES:  Never guess prices, timings, or delivery fees; use your available tools when needed.
FORMAT: Under 60 words, friendly."""

broken_agent = create_agent(
    model=llm,
    tools=[tool1, is_store_open, check_delivery_fee, log_stationery_order],
    system_prompt=SYSTEM_GENERIC,
)
broken_res = invoke_retry(broken_agent, {"messages": [{"role": "user", "content": "How much are 3 spiral notebooks?"}]})

print("=== Broken Agent Trace (tool1 / 'does stuff') ===")
for m in broken_res["messages"]:
    m.pretty_print()

broken_tool_calls = [
    tc["name"]
    for m in broken_res["messages"] if isinstance(m, AIMessage) and m.tool_calls
    for tc in m.tool_calls
]
print(f"\nTool calls made by broken agent: {broken_tool_calls}")
print("Observation: With name='tool1' and docstring='does stuff', the model did NOT pick tool1 to look up the price!")

=== Broken Agent Trace (tool1 / 'does stuff') ===
================================ Human Message =================================

How much are 3 spiral notebooks?
================================== Ai Message ==================================

I’m not sure of the current price. Could you let me know the price per spiral notebook, or would you like me to check the catalogue for you?

Tool calls made by broken agent: []
Observation: With name='tool1' and docstring='does stuff', the model did NOT pick tool1 to look up the price!


In [12]:
# Change it back to the properly named and documented get_stationery_price tool
restored_agent = create_agent(
    model=llm,
    tools=[get_stationery_price, is_store_open, check_delivery_fee, log_stationery_order],
    system_prompt=SYSTEM_GENERIC,
)
restored_res = invoke_retry(restored_agent, {"messages": [{"role": "user", "content": "How much are 3 spiral notebooks?"}]})

print("=== Restored Agent Trace (get_stationery_price with clear docstring) ===")
for m in restored_res["messages"]:
    m.pretty_print()

restored_tool_calls = [
    tc["name"]
    for m in restored_res["messages"] if isinstance(m, AIMessage) and m.tool_calls
    for tc in m.tool_calls
]
print(f"\nTool calls made by restored agent: {restored_tool_calls}")
assert "get_stationery_price" in restored_tool_calls, "Expected restored agent to call get_stationery_price!"
print("✅ Verified: Restoring the descriptive name and docstring fixed tool selection.")

=== Restored Agent Trace (get_stationery_price with clear docstring) ===
================================ Human Message =================================

How much are 3 spiral notebooks?
================================== Ai Message ==================================
Tool Calls:
  get_stationery_price (fc_57f6fb9c-d65e-4a9c-9d99-4aca138bde3d)
 Call ID: fc_57f6fb9c-d65e-4a9c-9d99-4aca138bde3d
  Args:
    item: spiral notebook (200 pgs)
    qty: 3
================================= Tool Message =================================
Name: get_stationery_price

3 x spiral notebook (200 pgs) @ Rs 85 = Rs 255
================================== Ai Message ==================================

3 spiral notebooks (200 pgs) cost Rs 255 in total. Let me know if you’d like to place an order or need anything else!

Tool calls made by restored agent: ['get_stationery_price']
✅ Verified: Restoring the descriptive name and docstring fixed tool selection.
